# Healthcare Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
df = pd.read_csv('dataset/healthcare_dataset.csv')
df.head()

## Data Cleaning

In [ ]:
# fix column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# fix name formatting
df['name'] = df['name'].str.title()

# convert dates
df['date_of_admission'] = pd.to_datetime(df['date_of_admission'])
df['discharge_date'] = pd.to_datetime(df['discharge_date'])

# calculate length of stay
df['length_of_stay'] = (df['discharge_date'] - df['date_of_admission']).dt.days

df.info()

## Demographics Analysis

In [ ]:
# age distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['age'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')
axes[0].set_title('Age Distribution')

df['gender'].value_counts().plot(kind='bar', ax=axes[1], color=['#5DA5DA', '#FAA43A'])
axes[1].set_title('Gender Distribution')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('results/plots/demographics.png', dpi=150)
plt.show()

## Medical Conditions

In [ ]:
# medical conditions breakdown
fig, ax = plt.subplots(figsize=(10, 5))

condition_counts = df['medical_condition'].value_counts()
bars = ax.bar(condition_counts.index, condition_counts.values, edgecolor='black')

ax.set_xlabel('Medical Condition')
ax.set_ylabel('Number of Patients')
ax.set_title('Patient Distribution by Medical Condition')
plt.xticks(rotation=45)

for bar, count in zip(bars, condition_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50, 
            str(count), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('results/plots/conditions.png', dpi=150)
plt.show()

## Billing Analysis

In [ ]:
# billing by condition
fig, ax = plt.subplots(figsize=(10, 5))

billing_by_condition = df.groupby('medical_condition')['billing_amount'].mean().sort_values(ascending=False)

bars = ax.barh(billing_by_condition.index, billing_by_condition.values)
ax.set_xlabel('Average Billing Amount ($)')
ax.set_title('Average Billing by Medical Condition')

for bar in bars:
    width = bar.get_width()
    ax.text(width + 500, bar.get_y() + bar.get_height()/2, 
            f'${width:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('results/plots/billing_by_condition.png', dpi=150)
plt.show()

## Insurance Providers

In [ ]:
# insurance provider distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# pie chart
insurance_counts = df['insurance_provider'].value_counts()
axes[0].pie(insurance_counts.values, labels=insurance_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Insurance Provider Distribution')

# avg billing by insurance
billing_by_insurance = df.groupby('insurance_provider')['billing_amount'].mean().sort_values()
axes[1].barh(billing_by_insurance.index, billing_by_insurance.values, color='steelblue')
axes[1].set_xlabel('Average Billing ($)')
axes[1].set_title('Average Billing by Insurance Provider')

plt.tight_layout()
plt.savefig('results/plots/insurance_analysis.png', dpi=150)
plt.show()

## Admission Type & Length of Stay

In [ ]:
# admission type breakdown
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

admission_counts = df['admission_type'].value_counts()
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']
axes[0].pie(admission_counts.values, labels=admission_counts.index, autopct='%1.1f%%', colors=colors)
axes[0].set_title('Admission Type Distribution')

# length of stay by admission type
sns.boxplot(data=df, x='admission_type', y='length_of_stay', ax=axes[1], palette='Set2')
axes[1].set_xlabel('Admission Type')
axes[1].set_ylabel('Length of Stay (days)')
axes[1].set_title('Length of Stay by Admission Type')

plt.tight_layout()
plt.savefig('results/plots/admission_analysis.png', dpi=150)
plt.show()

## Correlation Analysis

In [ ]:
# correlation heatmap for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', square=True, ax=ax)
ax.set_title('Correlation Matrix')

plt.tight_layout()
plt.savefig('results/plots/correlation_heatmap.png', dpi=150)
plt.show()

## Key Findings

- Age distribution is fairly uniform across patient population
- Medical conditions are evenly distributed (Cancer, Diabetes, Obesity, Asthma, Hypertension, Arthritis)
- Average billing ranges from ~$25k-26k across conditions
- Insurance providers have roughly equal market share
- Emergency admissions tend to have slightly longer stays
- Weak correlation between age and billing amount

In [ ]:
# summary stats
print(f"Total patients: {len(df):,}")
print(f"Date range: {df['date_of_admission'].min().date()} to {df['date_of_admission'].max().date()}")
print(f"Average billing: ${df['billing_amount'].mean():,.2f}")
print(f"Average length of stay: {df['length_of_stay'].mean():.1f} days")